# 07 - AutoGluon TabularPredictor

Trains AutoGluon's TabularPredictor on the same engineered features and
compares against the manually-trained models.


In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))
import pandas as pd
from src.utils import get_config, setup_logging
from src.models import ClassificationTrainer
setup_logging()
cfg = get_config()
X_tr = pd.read_csv(cfg.processed_data_dir / "X_train.csv")
y_tr = pd.read_csv(cfg.processed_data_dir / "y_train.csv").iloc[:,0]
from src.features import FeatureEngineer, FeatureSelector
eng = FeatureEngineer.load(cfg.preprocessing_dir / "feature_engineer.joblib")
sel = __import__("joblib").load(cfg.features_dir / "feature_selector.joblib")
X_tr_e = sel.transform(eng.transform(X_tr))
print(f"Train: {X_tr_e.shape}")


## 1. Train AutoGluon

In [ ]:
trainer = ClassificationTrainer(params_cfg=cfg.params, model_cfg=cfg.model)
ag = trainer.train_autogluon(X_tr_e, y_tr, cfg.autogluon_dir)
print(f"AutoGluon artifact: {cfg.autogluon_dir}")


## 2. Leaderboard

In [ ]:
if ag is not None:
    lb = ag.leaderboard()
    print(lb.head(15))
else:
    print("AutoGluon not available")


## 3. Compare to manual models

In [ ]:
import pandas as pd
cmp = pd.read_csv(cfg.evaluations_dir / "model_comparison.csv")
print(cmp[['model','recall_phishing','roc_auc','f1']]
      .sort_values('recall_phishing', ascending=False))


## Summary

AutoGluon trains a stack of base learners (LightGBM, CatBoost, XGBoost,
Random Forest, Neural Net, etc.) and combines them via stacking. When
installed, it provides a strong baseline that often matches or exceeds
hand-tuned ensembles - but its training time is significantly longer,
so it is treated as an optional comparison rather than the primary model.
